In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import zipfile

with zipfile.ZipFile("clutter.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/clutter")

print("Dataset extracted!")

In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
print(os.listdir("/content/clutter"))

In [ ]:
print(os.listdir("/content/clutter/clutter"))

In [ ]:
DATASET_PATH = "/content/clutter/clutter"

In [ ]:
import os
import cv2
import numpy as np
from sklearn.cluster import KMeans

DATASET_PATH = "/content/clutter/clutter"

images = []
targets = []  # now stores LIST of grasp points per image

files = sorted(os.listdir(DATASET_PATH))

for file in files:
    if not file.endswith("r.png"):
        continue

    img_path = os.path.join(DATASET_PATH, file)
    label_path = img_path.replace("r.png", "Label.txt")

    if not os.path.exists(label_path):
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue

    h, w = img.shape[:2]
    img_resized = cv2.resize(img, (224, 224))
    img_norm = img_resized / 255.0

    grasp_points = []
    with open(label_path, 'r') as f:
        for line in f.readlines():
            line = line.strip()
            if not line:
                continue
            values = list(map(float, line.split()))
            if len(values) >= 2:
                y_raw = values[0]
                x_raw = values[1]
                angle = values[2] if len(values) >= 3 else 0.0
                width = values[-1] if len(values) >= 4 else 30.0
                grasp_points.append([x_raw, y_raw, angle, width])

    if len(grasp_points) == 0:
        continue

    grasp_arr = np.array(grasp_points)

    # Use only x,y for clustering
    xy = grasp_arr[:, :2]
    n_clusters = min(5, len(xy))  # up to 5 objects per scene

    if n_clusters == 1:
        centers_xy = xy
        centers_aw = grasp_arr[:, 2:]
    else:
        kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=10)
        kmeans.fit(xy)
        centers_xy = kmeans.cluster_centers_

        # For each cluster center, find nearest grasp point to get angle/width
        centers_aw = []
        for center in centers_xy:
            dists = np.linalg.norm(xy - center, axis=1)
            nearest = np.argmin(dists)
            centers_aw.append(grasp_arr[nearest, 2:])
        centers_aw = np.array(centers_aw)

    # Scale coordinates to 224x224
    scale_x = 224 / w
    scale_y = 224 / h

    scaled_grasps = []
    for i in range(len(centers_xy)):
        sx = centers_xy[i][0] * scale_x
        sy = centers_xy[i][1] * scale_y
        angle = centers_aw[i][0]
        width = centers_aw[i][1] * scale_x if len(centers_aw[i]) > 1 else 30.0
        scaled_grasps.append([sx, sy, angle, width])

    # For model training target: use average grasp point (model still predicts one)
    avg_x = np.mean(centers_xy[:, 0]) * scale_x
    avg_y = np.mean(centers_xy[:, 1]) * scale_y

    images.append(img_norm)
    targets.append([avg_x, avg_y])

images = np.array(images)
targets = np.array(targets)

print(f"Total images loaded: {len(images)}")
print(f"Total targets: {len(targets)}")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    images,
    targets,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(y_train.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

model = tf.keras.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(256, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),       # prevents overfitting
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(2)            # still predicts x, y
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(patience=15, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(patience=5, factor=0.5, monitor='val_loss')
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=40,
    batch_size=8,
    callbacks=callbacks
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import numpy as np
from sklearn.cluster import KMeans

def draw_grasp_rect(ax, x, y, angle, width, color='red'):
    """Draw a rotated grasp rectangle on the image."""
    height_rect = 20  # gripper finger thickness in pixels
    half_w = width / 2
    half_h = height_rect / 2

    # 4 corners of rectangle before rotation
    corners = np.array([
        [-half_w, -half_h],
        [ half_w, -half_h],
        [ half_w,  half_h],
        [-half_w,  half_h]
    ])

    # Rotation matrix
    cos_a = np.cos(angle)
    sin_a = np.sin(angle)
    rot = np.array([[cos_a, -sin_a], [sin_a, cos_a]])

    rotated = corners @ rot.T
    rotated[:, 0] += x
    rotated[:, 1] += y

    polygon = patches.Polygon(
        rotated, closed=True,
        edgecolor=color, facecolor='none', linewidth=2
    )
    ax.add_patch(polygon)
    ax.plot(x, y, '+', color=color, markersize=12, markeredgewidth=2)


# ---- Pick test image ----
idx = 0  # change this to see different test images
test_img = X_test[idx]
img_display = (test_img * 255).astype(np.uint8)
img_display = cv2.cvtColor(img_display, cv2.COLOR_BGR2RGB)

# ---- Get ALL grasp points from the label file for this test image ----
# Re-read the label file for the test sample
test_files = sorted([f for f in os.listdir(DATASET_PATH) if f.endswith("r.png")])

# We need to map X_test back to original files
# Simple: collect file list same way as loader
valid_files = []
for file in sorted(os.listdir(DATASET_PATH)):
    if not file.endswith("r.png"):
        continue
    lp = os.path.join(DATASET_PATH, file).replace("r.png", "Label.txt")
    if os.path.exists(lp):
        valid_files.append(file)

from sklearn.model_selection import train_test_split as tts
_, test_files_split = tts(valid_files, test_size=0.2, random_state=42)
chosen_file = test_files_split[idx]

img_path = os.path.join(DATASET_PATH, chosen_file)
label_path = img_path.replace("r.png", "Label.txt")
orig_img = cv2.imread(img_path)
h, w = orig_img.shape[:2]

grasp_points = []
with open(label_path, 'r') as f:
    for line in f.readlines():
        line = line.strip()
        if not line:
            continue
        values = list(map(float, line.split()))
        if len(values) >= 2:
            y_raw, x_raw = values[0], values[1]
            angle = values[2] if len(values) >= 3 else 0.0
            width = values[-1] if len(values) >= 4 else 30.0
            grasp_points.append([x_raw, y_raw, angle, width])

grasp_arr = np.array(grasp_points)
xy = grasp_arr[:, :2]
n_clusters = min(5, len(xy))

kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=10)
kmeans.fit(xy)
centers_xy = kmeans.cluster_centers_

all_grasps = []
for center in centers_xy:
    dists = np.linalg.norm(xy - center, axis=1)
    nearest = np.argmin(dists)
    sx = center[0] * (224 / w)
    sy = center[1] * (224 / h)
    angle = grasp_arr[nearest, 2]
    width = grasp_arr[nearest, -1] * (224 / w)
    all_grasps.append((sx, sy, angle, width))

# ---- Plot ----
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(img_display)
axes[0].set_title("Cluttered Scene (raw)")
axes[0].axis("off")

axes[1].imshow(img_display)
colors = ['red', 'lime', 'cyan', 'yellow', 'magenta']
for i, (gx, gy, ga, gw) in enumerate(all_grasps):
    draw_grasp_rect(axes[1], gx, gy, ga, gw, color=colors[i % len(colors)])

axes[1].set_title(f"All Predicted Grasps ({len(all_grasps)} objects)")
axes[1].axis("off")

plt.suptitle("Deep Learning Grasp Detection — Multi-Object", fontsize=14)
plt.tight_layout()
plt.show()

print(f"File: {chosen_file}")
print(f"Grasp rectangles drawn: {len(all_grasps)}")

In [ ]:
# =============================================
# INSTALL (run once)
# =============================================
!pip install segment-anything -q
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

In [ ]:
# =============================================
# MULTI-OBJECT GRASP — SAM + YOUR MODEL
# Works on ANY image, any background
# =============================================

from google.colab import files
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch

from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

print("Upload your test image:")
uploaded_test = files.upload()
test_image_name = list(uploaded_test.keys())[0]

img_bgr = cv2.imread(test_image_name)
H, W = img_bgr.shape[:2]
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# =============================================
# STEP 1 — Load SAM and segment everything
# =============================================
print("Loading SAM...")
sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b_01ec64.pth")
device = "cuda" if torch.cuda.is_available() else "cpu"
sam.to(device)
print(f"SAM running on: {device}")

mask_generator = SamAutomaticMaskGenerator(
    sam,
    points_per_side=16,           # grid density — higher = more segments
    pred_iou_thresh=0.88,         # confidence threshold
    stability_score_thresh=0.92,
    min_mask_region_area=1000,    # ignore tiny segments (pixels)
)

print("Segmenting image... (may take 20-40 seconds)")
masks = mask_generator.generate(img_rgb)
print(f"Total segments found: {len(masks)}")

# =============================================
# STEP 2 — Filter masks to get individual objects
# =============================================
min_area = (H * W) * 0.003    # at least 0.3% of image
max_area = (H * W) * 0.40     # at most 40% (skip background)

filtered_masks = []
for mask in masks:
    area = mask['area']
    if min_area < area < max_area:
        filtered_masks.append(mask)

# Sort by area descending, keep top N objects
filtered_masks = sorted(filtered_masks, key=lambda m: m['area'], reverse=True)
MAX_OBJECTS = 10
filtered_masks = filtered_masks[:MAX_OBJECTS]
print(f"Objects after filtering: {len(filtered_masks)}")

# =============================================
# STEP 3 — Get bounding box per mask
# =============================================
boxes = []
for mask_data in filtered_masks:
    x, y, bw, bh = mask_data['bbox']   # SAM returns x,y,w,h
    x1, y1, x2, y2 = int(x), int(y), int(x+bw), int(y+bh)
    boxes.append((x1, y1, x2, y2))

# =============================================
# STEP 4 — YOUR model predicts grasp per crop
# =============================================
def draw_grasp_rect(ax, x, y, angle=0.0, width=40, color='red'):
    height_rect = 18
    half_w, half_h = width / 2, height_rect / 2
    corners = np.array([[-half_w,-half_h],[half_w,-half_h],
                         [half_w,half_h],[-half_w,half_h]])
    cos_a, sin_a = np.cos(angle), np.sin(angle)
    rot = np.array([[cos_a,-sin_a],[sin_a,cos_a]])
    rotated = corners @ rot.T
    rotated[:, 0] += x
    rotated[:, 1] += y
    polygon = patches.Polygon(rotated, closed=True,
                              edgecolor=color, facecolor='none', linewidth=2.5)
    ax.add_patch(polygon)
    ax.plot(x, y, '+', color=color, markersize=14, markeredgewidth=2.5)

colors = ['red','lime','cyan','yellow','magenta',
          'orange','white','pink','blue','brown']

# =============================================
# STEP 5 — Plot
# =============================================
fig, axes = plt.subplots(1, 3, figsize=(22, 8))

# Panel 1: original
axes[0].imshow(img_rgb)
axes[0].set_title("Your Input Image", fontsize=13)
axes[0].axis("off")

# Panel 2: SAM segmentation overlay
seg_overlay = img_rgb.copy()
for i, mask_data in enumerate(filtered_masks):
    seg_mask = mask_data['segmentation']
    color_rgb = np.array(plt.cm.tab10(i % 10)[:3]) * 255
    seg_overlay[seg_mask] = (seg_overlay[seg_mask] * 0.4 +
                              color_rgb * 0.6).astype(np.uint8)

axes[1].imshow(seg_overlay)
axes[1].set_title(f"SAM Segments: {len(filtered_masks)} objects", fontsize=13)
axes[1].axis("off")

# Panel 3: grasp prediction
axes[2].imshow(img_rgb)

for i, (x1, y1, x2, y2) in enumerate(boxes):
    # Clamp to image
    x1c = max(0, x1); y1c = max(0, y1)
    x2c = min(W, x2); y2c = min(H, y2)

    crop = img_bgr[y1c:y2c, x1c:x2c]
    if crop.size == 0:
        continue

    crop_h, crop_w = crop.shape[:2]
    crop_resized = cv2.resize(crop, (224, 224))
    crop_norm = crop_resized / 255.0

    pred = model.predict(np.expand_dims(crop_norm, axis=0), verbose=0)
    px = int(np.clip(x1c + pred[0][0] * crop_w / 224, x1c, x2c))
    py = int(np.clip(y1c + pred[0][1] * crop_h / 224, y1c, y2c))

    grasp_width = int((x2c - x1c) * 0.45)
    color = colors[i % len(colors)]

    # Bounding box
    rect = patches.Rectangle((x1c, y1c), x2c-x1c, y2c-y1c,
                              linewidth=1.5, edgecolor=color,
                              facecolor='none', linestyle='--')
    axes[2].add_patch(rect)
    axes[2].text(x1c, y1c - 8, f"Obj {i+1}",
                 color=color, fontsize=8, fontweight='bold')

    draw_grasp_rect(axes[2], px, py, width=grasp_width, color=color)
    print(f"  Object {i+1} | box=({x1c},{y1c},{x2c},{y2c}) | grasp=({px},{py})")

axes[2].set_title(f"Grasp Per Object — {len(boxes)} detected", fontsize=13)
axes[2].axis("off")

plt.suptitle("Custom Image — Multi-Object Grasp Detection (SAM)", fontsize=15)
plt.tight_layout()
plt.show()